In [3]:
#%pip install rdkit-pypi requests pandas

In [4]:
import requests
import pandas as pd
from rdkit import RDLogger, Chem

RDLogger.DisableLog("rdApp.*")

# BindingDB REST API — use UniProt ID for HMGCR (P04035)
url = "https://bindingdb.org/rest/getLigandsByUniprot"
params = {
    "uniprot":     "P04035",
    "response":    "application/json"
}

print("Fetching from BindingDB...")
response = requests.get(url, params=params, timeout=120)
print(f"Status: {response.status_code}")

if response.status_code != 200:
    raise RuntimeError(
        f"BindingDB returned HTTP {response.status_code}. "
        f"Response body: {response.text[:500]}"
    )

data       = response.json()
affinities = data.get("getLigandsByUniprot", {}).get("affinities", [])
print(f"Raw entries: {len(affinities)}")

rows = []
for entry in affinities:
    rows.append({
        "smiles":   entry.get("smile", ""),
        "affinity": entry.get("affinity", ""),
        "units":    entry.get("affinity_type", ""),
        "target":   entry.get("target_name", ""),
    })

df_bdb = pd.DataFrame(rows)

# clean SMILES
def clean_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi).strip())
        if mol is None: return None
        return Chem.MolToSmiles(mol)
    except:
        return None

df_bdb["clean_smiles"] = df_bdb["smiles"].apply(clean_smiles)
df_bdb = df_bdb.dropna(subset=["clean_smiles"]).drop_duplicates(subset=["clean_smiles"])
print(f"Valid unique SMILES: {len(df_bdb)}")
print(df_bdb.head())

Fetching from BindingDB...
Status: 200
Raw entries: 0


KeyError: 'smiles'